In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
import yaml

In [2]:
with (Path.cwd().parent / "datasets.yaml").open() as file:
    dataset_mapping = yaml.safe_load(file)

In [3]:
DATA_DIR = Path('/home/casadoj/Git/ml-benchmark/data')

In [4]:
MODEL = 'DeepESD-IFCAv1'
DOMAIN = 'ALPS'
TRAIN_EXP = 'ESD_pseudo_reality'
RCM = 'CNRM-ALADIN63'
TRAIN_GCM = 'CNRM-CM5'
DATASET = 'CNRM-CM5'
PERIOD = '1981-2000'
SEASON = 'annual'
VAR = 'tasmax'

In [38]:
target_value = {
    'rmse': {
        'ESD_pseudo_reality': {
            '1981-2000': 1.50,
            '2041-2060': 2.55,
        },
        'Emulator_hist_future': {
                '1981-2000': 1.39,
                '2041-2060': 1.42,
        }
    },
    'bias_mean': {
        'ESD_pseudo_reality': {
            '1981-2000': 0.22,
            '2041-2060': 1.34,
        },
        'Emulator_hist_future': {
                '1981-2000': 0.21,
                '2041-2060': 0.25,
        }
    },
    'bias_max1day': {
        'ESD_pseudo_reality': {
            '1981-2000': 0.93,
            '2041-2060': 2.21,
        },
        'Emulator_hist_future': {
                '1981-2000': 0.84,
                '2041-2060': 0.86,
        }
    },
    'ralsd': {
        'ESD_pseudo_reality': {
            '1981-2000': 6.41,
            '2041-2060': 7.61,
        },
        'Emulator_hist_future': {
                '1981-2000': 4.73,
                '2041-2060': 4.63,
        }
    },
}

In [50]:
zarr_file.is_file()

False

In [53]:
metric = 'bias_max1day'

print(metric.upper())
print('-' * len(metric))

for train_exp in ['ESD_pseudo_reality', 'Emulator_hist_future']:
    for period in ['1981-2000', '2041-2060']:
        print(f'\n| {train_exp} | {period} |')
        metric_float = []
        metric_field = []
        for domain in ['ALPS', 'NZ', 'SA']:
            # define evaluation path
            rcm, train_gcm, test_gcm = dataset_mapping[domain].values()
            path = DATA_DIR / MODEL / 'evaluation' / f'{domain}_Domain' / train_exp / f'{rcm}_{train_gcm}_{train_gcm}_{period}' / SEASON / VAR
            if not path.is_dir():
                print(f'Path does not exist: {path}')
                continue

            # extract metric computed over time and space
            zarr_file = path / 'metrics_fast_float.zarr'
            if not zarr_file.is_dir():
                print(f'File not found: {zarr_file}')
            fast_float = xr.open_zarr(zarr_file)['value'].load()
            try:
                metric_float.append(fast_float.sel(diagnostic=f'{metric}_float').item())
            except Exception as e:
                print(e)
                pass
                print(f'{metric} not included in the fast-evaluation floats.')

            # extract RMSE computed over time, and then aggregated over space
            zarr_file = path / 'metrics_fast_field.zarr'
            if not zarr_file.is_dir():
                print(f'File not found: {zarr_file}')
            fast_field = xr.open_zarr(zarr_file)['value'].load()
            try:
                metric_field.append(np.abs(fast_field.sel(diagnostic=f'{metric}_field')).mean().item())
            except Exception as e:
                print(e)
                pass
                
        target = target_value[metric][train_exp][period]
        print(f'{metric.upper()} (target) = {target:.2f}')
        if len(metric_float) > 0:
            print(f'{metric.upper()} (float)  = {np.mean(metric_float):.2f}')
        if len(metric_field) > 0:
            print(f'{metric.upper()} (field)  = {np.mean(metric_field):.2f}')

BIAS_MAX1DAY
------------

| ESD_pseudo_reality | 1981-2000 |
"not all values found in index 'diagnostic'. Try setting the `method` keyword argument (example: method='nearest')."
bias_max1day not included in the fast-evaluation floats.
"not all values found in index 'diagnostic'. Try setting the `method` keyword argument (example: method='nearest')."
bias_max1day not included in the fast-evaluation floats.
"not all values found in index 'diagnostic'. Try setting the `method` keyword argument (example: method='nearest')."
bias_max1day not included in the fast-evaluation floats.
BIAS_MAX1DAY (target) = 0.93
BIAS_MAX1DAY (field)  = 0.68

| ESD_pseudo_reality | 2041-2060 |
"not all values found in index 'diagnostic'. Try setting the `method` keyword argument (example: method='nearest')."
bias_max1day not included in the fast-evaluation floats.
"not all values found in index 'diagnostic'. Try setting the `method` keyword argument (example: method='nearest')."
bias_max1day not included in th

The values in the paper correspond in general to the float value in `ml-evaluation` averaged across domains.

> <font color='maroon'>**WARNING!**</font>. The value in the *ESD_pseudo_reality* experiment and period 2041-2060 is off compared with the two approximations.

In [20]:
fast_float['diagnostic']

<xarray.DataArray 'diagnostic' (diagnostic: 2)> Size: 88B
array(['rmse_float', 'ralsd_float'], dtype='<U11')
Coordinates:
  * diagnostic  (diagnostic) <U11 88B 'rmse_float' 'ralsd_float'

In [27]:
fast_field['diagnostic']

<xarray.DataArray 'diagnostic' (diagnostic: 22)> Size: 2kB
array(['rmse_field', 'bias_mean_field', 'bias_p99_field', 'bias_p1_field',
       'bias_acorr_field', 'bias_std_field', 'bias_max1day_field',
       'bias_min1day_field', 'targ_mean_field', 'pred_mean_field',
       'targ_p99_field', 'pred_p99_field', 'targ_p1_field', 'pred_p1_field',
       'targ_acorr_field', 'pred_acorr_field', 'targ_std_field',
       'pred_std_field', 'targ_max1day_field', 'pred_max1day_field',
       'targ_min1day_field', 'pred_min1day_field'], dtype='<U18')
Coordinates:
  * diagnostic  (diagnostic) <U18 2kB 'rmse_field' ... 'pred_min1day_field'
    quantile    float64 8B 0.99

BIAS_MAX1DAY
------------


| ESD_pseudo_reality | 1981-2000 |
BIAS_MAX1DAY (target) = 0.93

| ESD_pseudo_reality | 2041-2060 |
BIAS_MAX1DAY (target) = 2.21

| Emulator_hist_future | 1981-2000 |
BIAS_MAX1DAY (target) = 0.84

| Emulator_hist_future | 2041-2060 |
BIAS_MAX1DAY (target) = 0.86


In [37]:
metric = 'bias_mean'

for train_exp in ['ESD_pseudo_reality', 'Emulator_hist_future']:
    for period in ['1981-2000', '2041-2060']:
        metric_float = []
        metric_field = []
        for domain in ['ALPS', 'NZ', 'SA']:
            # define evaluation path
            rcm, train_gcm, test_gcm = dataset_mapping[domain].values()
            path = DATA_DIR / MODEL / 'evaluation' / f'{domain}_Domain' / train_exp / f'{rcm}_{train_gcm}_{train_gcm}_{period}' / SEASON / VAR
            if not path.is_dir():
                print(f'Path does not exist: {path}')
                continue

            # extract metric computed over time and space
            try:
                fast_float = xr.open_zarr(path / 'metrics_fast_float.zarr')['value'].load()
                metric_float.append(fast_float.sel(diagnostic=f'{metric}_float').item())
            except:
                print(f'{metric} not included in the fast-evaluation floats.')

            # extract RMSE computed over time, and then aggregated over space
            try:
                fast_field = xr.open_zarr(path / 'metrics_fast_field.zarr')['value'].load()
                metric_field.append(np.abs(fast_field.sel(diagnostic=f'{metric}_field')).mean().item())
            except:
                print(f'{metric} not included in the fast-evaluation fields.')
                
        print(f'| {train_exp} | {period} |')
        target = target_value[metric][train_exp][period]
        print(f'{metric.upper()} (target) = {target:.2f}')
        print(f'{metric.upper()} (float)  = {np.mean(metric_float):.2f}')
        print(f'{metric.upper()} (field)  = {np.mean(metric_field):.2f}\n')

bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
| ESD_pseudo_reality | 1981-2000 |
BIAS_MEAN (target) = 0.22
BIAS_MEAN (float)  = nan
BIAS_MEAN (field)  = 0.18

bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.


/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


bias_mean not included in the fast-evaluation floats.
| ESD_pseudo_reality | 2041-2060 |
BIAS_MEAN (target) = 1.34
BIAS_MEAN (float)  = nan
BIAS_MEAN (field)  = 0.61

bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
| Emulator_hist_future | 1981-2000 |
BIAS_MEAN (target) = 0.21
BIAS_MEAN (float)  = nan
BIAS_MEAN (field)  = 0.18

bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
bias_mean not included in the fast-evaluation floats.
| Emulator_hist_future | 2041-2060 |
BIAS_MEAN (target) = 0.25
BIAS_MEAN (float)  = nan
BIAS_MEAN (field)  = 0.21



/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/casadoj/miniconda3/envs/rcm-emulation/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [32]:
fast_field.sel(diagnostic=f'{metric}_field').abs().mean().item()

AttributeError: 'DataArray' object has no attribute 'abs'

In [43]:
fast_field = xr.open_zarr(path / 'metrics_fast_field.zarr')['value'].load()
fast_field.sel(diagnostic='rmse_field').mean().item()

1.6474065690272255

In [35]:
fast_field.sel(diagnostic='rmse_field').mean().item()

1.6474065690272255

In [15]:
slow_field = xr.open_zarr(path / 'metrics_slow_field.zarr')['value'].load()
slow_field['diagnostic']

<xarray.DataArray 'diagnostic' (diagnostic: 13)> Size: 884B
array(['bias_annstd_field', 'bias_p90_field', 'wd_field', 'wdh_field',
       'fcss_field', 'pss_field', 'lhd_field', 'pcorr_field', 'scorr_field',
       'targ_annstd_field', 'pred_annstd_field', 'targ_p90_field',
       'pred_p90_field'], dtype='<U17')
Coordinates:
  * diagnostic  (diagnostic) <U17 884B 'bias_annstd_field' ... 'pred_p90_field'
    quantile    float64 8B 0.9

In [16]:
slow_tasmax_field = xr.open_zarr(path / 'metrics_slow_tasmax_field.zarr')['value'].load()
slow_tasmax_field['diagnostic']

<xarray.DataArray 'diagnostic' (diagnostic: 9)> Size: 504B
array(['bias_su_field', 'bias_hot_field', 'bias_ice_field', 'targ_su_field',
       'pred_su_field', 'targ_hot_field', 'pred_hot_field', 'targ_ice_field',
       'pred_ice_field'], dtype='<U14')
Coordinates:
  * diagnostic  (diagnostic) <U14 504B 'bias_su_field' ... 'pred_ice_field'